## Example of SDE-matching using a Gaussian Process as marginal distribution

We define a data generating process by the following SDE:
$$
dz_t = f(z_t, t) dt + g(z_t, t) dW_t = \begin{bmatrix}
0 & 2 \\
-2 & 0
\end{bmatrix} z_t d_t + 0.6 \cdot dWt
$$

with emission distribution $y_t \sim \mathcal N(y_t\vert z_t, 0.3^2)$ and initial conditions: $z_0 \sim \mathcal N(0, I_2)$. 


This model is a classic pendulum model. In this case we assume that we can only observe the position -- that is the first dimension. 

The task is to reconstruct the drift matrix $A=\begin{bmatrix}
0 & 2 \\
-2 & 0
\end{bmatrix}$, the diffusion constant $\sigma_d = 0.6$, the observation noise $\sigma_e = 0.3$ and the initial condition distribution $\mathcal N(0, I_2)$.

Note that since we only observe the first dimension, there are many SDE's that can generate the same observations. 
Indeed, if 
$A=\begin{bmatrix}
0 & 2 \\
-2 & 0
\end{bmatrix},$ then any matrix $A'=\begin{bmatrix}
a & b \\
c & d
\end{bmatrix}$ where trace and determinant are the same as for $A$ will generate the same process in the first dimension. That means that as long as $a+d=0$ and $ad - bc=4$ we will see the same observed process. In other words, there are two free parameters in the model. 

But in this specific case we know that the system is energy conserving, we can safely assume that the drift matrix is skew-diagonal. That is: $A'=\begin{bmatrix}
0 & p \\
-p & 0
\end{bmatrix}$.

In [ ]:
# First some imports
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import matplotlib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from SDEmatching.distributions.Emission import GaussianEmission
from SDEmatching.distributions.Prior import GaussianPrior
from SDEmatching.core.Diffusions import ScalarDiffusion, SimpleDiffusion
from SDEmatching.core.Marginal import Marginal
from SDEmatching.models.Flows import DDPMflow, NormalFlow, AffineFlow
from SDEmatching.core.SDE import SimpleSDE, SDE, manual_euler_sample
from SDEmatching.utils.utils import torch_seed, to_tensor, mask_and_pad, save_models
import seaborn as sns
from SDEmatching.utils.datageneration import SDEdatagenerator
from SDEmatching.core.SDEproblem import SDEproblem
from plot_functions import plot_parameter_history, plot_marginal
from SDEmatching.models.ConditionMappers import TransformerLatentModel
from SDEmatching.models.GaussianProcessDerivative import GPLatentModel, RBFKernel, PeriodicKernel
import math
import numpy as np
import normflows as nf
from torch.utils.data import DataLoader, TensorDataset
import copy
import pandas as pd
#torch.autograd.set_detect_anomaly(True)

In [ ]:
# set constants
num_timesteps = 30
num_ts_samples = 30
num_series = 200
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

state_dim = 2
observation_dim = 1
t_start = -1
t_end = 1
time_dist = torch.distributions.Uniform(t_start, t_end)

true_diffusion_log_std = torch.tensor(.6).log()
true_prior_log_std = (torch.ones(state_dim, device=device)).log()
true_emission_log_std = torch.tensor(.3).log()

In [ ]:
"""
Generating data
"""
# diffusion function  
generator_diffusion = SimpleDiffusion(brownian_dim=2, state_dim=state_dim, log_std=true_diffusion_log_std, device=device, trainable=True)

# drift function
class LinearDrift(torch.nn.Module):
    def __init__(self, state_dim, device='cpu'):
        super().__init__()
        self.state_dim = state_dim
        self.device    = device
        self.linear = torch.nn.Linear(state_dim, state_dim, bias=False).to(device)

    def forward(self, state, ts=None):
        return self.linear(state)

generator_drift = LinearDrift(state_dim=state_dim, device=device)
generator_drift.linear.weight.data = torch.tensor([[0., 2.], [-2., 0.]], device=device)

# prior
generator_prior = GaussianPrior(mean=torch.ones(state_dim, device=device) * 0., log_std=true_prior_log_std, device=device, trainable=True)

# emission distribution:
#generator_emission = GaussianEmission(dim=state_dim, log_std=true_emission_log_std, device=device, trainable=True)
generator_emission = GaussianEmission(dim=state_dim, observation_dim=observation_dim, log_std=true_emission_log_std, device=device, trainable=True)

# put together the SDE
generator_SDE = SDE(generator_drift, generator_diffusion, generator_prior, steps=num_timesteps, t_start=t_start, t_end=t_end, device=device)
#list_of_timeseries = SDEdatagenerator(generator_SDE, generator_emission, time_dist, num_series=11, mean_num_ts=10, same_ts=False, num_ts_samples=None, device=device)

# generate data
list_of_timeseries, list_of_state_timeseries = SDEdatagenerator(generator_SDE, generator_emission, time_dist, num_series=num_series, same_ts=False, mean_num_ts=num_ts_samples, device=device, seed=2)

# convert to tensors
true_state_series = torch.stack([state_timeseries for state_timeseries in list_of_state_timeseries]).detach().clone()

# The time series have different lengths, so we need pad them and define a mask to be able to use them in batches. 
data, emissions_mask = mask_and_pad(list_of_timeseries, list_of_state_timeseries, observation_dim, device)

In [ ]:
# Plotting some paths and emissions
cmap = matplotlib.colormaps["viridis"]
series_to_plot=min(5, data.shape[0])
for i in range(series_to_plot):
    color = cmap(i/series_to_plot)
    if i == 0:
        emmission_label = "emissions"
        state_label = "true state"
    else:        
        emmission_label = None
        state_label = None
    
    _=plt.plot(data[i, ~emissions_mask[i],0].T.detach().cpu(), data[i, ~emissions_mask[i],1].T.detach().cpu(), marker="x", linewidth=0, c=color, label=emmission_label)
    _=plt.plot(generator_SDE.ts.detach().cpu(), true_state_series[i,:,0].T.detach().cpu(), marker="", c=color, label=state_label)
    plt.xlabel("time")
    plt.title("True states and emissions")
    plt.legend()
plt.show()

## Define model setup for training
We define a model of the same class as the generating model:
$$
dz_t = f(z_t, t) dt + g(z_t, t) dW_t = A_\theta z_t d_t + \sigma_\theta \cdot dWt
$$

with emission distribution: $y_t \sim \mathcal N(y_t\vert z_t, \sigma^2_y)$ and initial conditions: $z_0 \sim \mathcal N(\mu_I, \sigma^2_I \cdot I_2)$. We use a special class for the skew-symmetric drift matrix $A$.

In [ ]:
class SkewSymmetricLinear(torch.nn.Module):
    """
    A linear layer restricted to skew-symmetric matrices: A = P - P^T
    Only the upper triangular entries are learnable; diagonal is zero
    and lower triangle is determined by skew-symmetry.
    
    For state_dim=2: A = [[0, p], [-p, 0]] with 1 parameter.
    """
    def __init__(self, state_dim, device='cpu'):
        super().__init__()
        self.state_dim = state_dim
        self.device    = device

        # Only upper triangle parameters
        n_upper = state_dim * (state_dim - 1) // 2
        self.upper_params = torch.nn.Parameter(
            torch.randn(n_upper, device=device) * 0.1
        )

        # Precompute upper triangle indices
        upper = torch.triu_indices(state_dim, state_dim, offset=1)
        self.row_idx = upper[0]
        self.col_idx = upper[1]

    def get_matrix(self):
        P = torch.zeros(self.state_dim, self.state_dim, device=self.device)
        P[self.row_idx, self.col_idx] = self.upper_params
        return P - P.T  # skew-symmetric by construction

    def forward(self, state, ts=None):
        return state @ self.get_matrix().T

In [ ]:
# Prior
priormean = torch.zeros(state_dim, device=device)
prior_log_std = true_prior_log_std.clone().detach()
myprior = GaussianPrior(mean=priormean, log_std=prior_log_std, trainable=True, device=device)

# Diffusion term
mydiffusion = SimpleDiffusion(state_dim, state_dim, log_std=true_diffusion_log_std.clone().detach(), device=device, trainable=True)

# Drift term
mydrift = SkewSymmetricLinear(state_dim=state_dim, device=device)

# Emission distribution:
myemission = GaussianEmission(dim=state_dim, observation_dim=observation_dim, log_std=true_emission_log_std.clone().detach(), trainable=True, device=device)


## Building the marginal distribution

In this setup, we use a special Gaussian Process, which both output the mean and variance, but *also* output the derivative of the mean, and the variance of the derivative of the mean. It can do so for all input dimensions, but here we only have one input dimension, so the output will be a stacked tensor of the form: `[mu, dot_mu, log_sigma, log_dot_sigma]`.

The GP process with derivative is chosen since the Transformer model has much difficulties recreating the hidden second dimension of the SDE model.  

In [ ]:
kernel = RBFKernel(obs_dim=observation_dim, device=device).to(device)
condition_mapper = GPLatentModel(obs_dim=observation_dim, device=device, kernel=kernel, linearmapping=False).to(device)
marginal_func = AffineFlow(state_dim, device=device)
myMarginal = Marginal(marginal_func, mydiffusion, condition_mapper, device=device)

mySDEproblem = SDEproblem(drift=mydrift, 
                          diffusion=mydiffusion, 
                          prior=myprior, 
                          marginal_func=marginal_func, 
                          condition_mapper=condition_mapper, 
                          emission=myemission, 
                          time_dist=time_dist, 
                          t_start=t_start, 
                          t_end=t_end, 
                          device=device)

### Model parameters
This model has surprisingly few parameters, so we can list all of them here. Note that the Gaussian process (`condition_mapper`) has a few hyper parameters which we also optimize.



In [ ]:

print(f"Total parameters in mySDEproblem: {sum(p.numel() for p in mySDEproblem.parameters())}")
for k in dict(mySDEproblem.named_parameters()).keys():
    print(k)

print("\n\nGenerating modules parameters")
for net in [generator_drift, generator_diffusion, generator_emission, generator_prior]:
    print(f"\n{net._get_name()}:")
    print(dict(net.named_parameters()))

print("\n\nSolving modules parameters")
for net in [mydrift, mydiffusion, myemission, myprior, myMarginal]:
    print(f"\n{net._get_name()}:")
    print(dict(net.named_parameters()))

## Marginal distribution before training
Let's have a look at the marginal distribution for three samples of time series. As we can see, the maringal is not completely off, as is the case when we use the Transformer model. 

In [ ]:
# Plotting the marginal distribution as it looks now
fig, axes = plot_marginal(mySDEproblem, data, state_dim, axes=None, true_states=true_state_series, 
                          true_states_ts=generator_SDE.ts, num_samples=100, max_num_data=3, device=device, observation_dim=observation_dim, data_mask=emissions_mask)
plt.show()


In [ ]:
# train loop
saved_parameter_dict = {}

all_train_losses = []
batch_size = 100
max_step_no = 1520
with torch_seed(0):
    #train_loader_ = DataLoader(data, batch_size=batch_size)
    dataset = TensorDataset(data, emissions_mask, true_state_series)
    train_loader_ = DataLoader(dataset, batch_size=batch_size, shuffle=True)
epoch_loss = []
epoch_diffusion_loss = []
epoch_prior_loss = []
epoch_reconstruction_loss = []

# saving the true parameters for later comparison
true_parameter_dict ={}
save_models(true_parameter_dict, [generator_drift, generator_diffusion, generator_emission, generator_prior])

# list of models for plotting 
models_to_be_saved = [mydrift, mydiffusion, myemission, myprior]
saved_models = []

num_batches_train = len(train_loader_)
epoch_steps = []
step_no = 0
epoch = 0
plot_every_epoch = 5000
plot_parameters_every_epoch = 5000
print_every_epoch = 10

# We want to train in two tempi, since the training of the marginal otherwise becomes unstable.
params_for_optim1 = [{"params": m.named_parameters()} for m in [mydrift, mydiffusion, myemission, myprior] if len(dict(m.named_parameters()))>0]
params_for_optim2 = mySDEproblem.marginal.named_parameters()
diffusion_weight = 1
lr1 = torch.tensor(0.2)
lr2 = torch.tensor(0.004)
optimizer1 = torch.optim.Adam(params_for_optim1, lr=lr1)
optimizer2 = torch.optim.Adam(params_for_optim2, lr=lr2)
scheduler=None

In [ ]:
while True:
    epoch_loss.append(0) 
    epoch_diffusion_loss.append(0) 
    epoch_prior_loss.append(0) 
    epoch_reconstruction_loss.append(0) 

    epoch += 1
    save_models(saved_parameter_dict, models_to_be_saved)

    for data_batch, mask_batch, states_batch in train_loader_:
        this_batch_size = len(data_batch)
        mySDEproblem.train()
        optimizer1.zero_grad()
        optimizer2.zero_grad()
        diffusion_loss, prior_loss, reconstruction_loss = mySDEproblem.ELBO(data_batch, mask_batch)
        sum_loss = (diffusion_loss*diffusion_weight + prior_loss*1 + reconstruction_loss).mean(dim=0)
        sum_loss.backward()
        all_train_losses.append(sum_loss.item())
        epoch_loss[-1] += sum_loss.item() / num_batches_train
        epoch_diffusion_loss[-1] += diffusion_loss.mean(dim=0).item() / num_batches_train
        epoch_prior_loss[-1] += prior_loss.mean(dim=0).item() / num_batches_train
        epoch_reconstruction_loss[-1] += reconstruction_loss.mean(dim=0).item() / num_batches_train
        step_no += 1

        optimizer1.step()
        optimizer2.step()

    if (epoch % print_every_epoch) ==0:
        string1 = f"epoch={len(epoch_loss)}, \tstep={step_no}, "
        string2 = f"\tloss={sum(epoch_loss[-print_every_epoch:])/print_every_epoch :2.6f}, \tdiff loss={sum(epoch_diffusion_loss[-print_every_epoch:])/print_every_epoch :2.6f}, "
        string3 = f"\tprior loss={sum(epoch_prior_loss[-print_every_epoch:])/print_every_epoch :2.6f}, \trec loss={sum(epoch_reconstruction_loss[-print_every_epoch:])/print_every_epoch :2.6f},"
        print(string1 + string2 + string3)

    if (epoch % plot_every_epoch)==0:
        fig, axes = plot_marginal(mySDEproblem, data, state_dim, axes=None, true_states=true_state_series, 
                          true_states_ts=generator_SDE.ts, num_samples=100, num_timesteps = 15, max_num_data=3, device=device, observation_dim=observation_dim, data_mask=emissions_mask)
        plt.show()

    if (epoch % plot_parameters_every_epoch)==0:
        fig, axes = plot_parameter_history(true_parameter_dict, saved_parameter_dict, step_no)
        plt.show()
        #pass

    if scheduler:
        scheduler.step()
    if step_no >= max_step_no: break


### Parameters after training

In [ ]:
print(f"Total parameters in mySDEproblem: {sum(p.numel() for p in mySDEproblem.parameters())}")
for k in dict(mySDEproblem.named_parameters()).keys():
    print(k)

print("\n\nGenerating modules parameters")
for net in [generator_drift, generator_diffusion, generator_emission, generator_prior]:
    print(f"\n{net._get_name()}:")
    print(dict(net.named_parameters()))

print("\n\nSolving modules parameters")
for net in [mydrift, mydiffusion, myemission, myprior, condition_mapper, marginal_func]:
    print(f"\n{net._get_name()}:")
    print(dict(net.named_parameters()))

# Conclusion
As we can see here, the model actually converges to the correct parameters except for the diffusion constant which is around half of what we would want. 

In [ ]:

all_train_losses_t = torch.tensor(all_train_losses)

def moving_average(a, n=3):
    ret = a.cumsum(dim=0)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n

train_loss_ma = moving_average(all_train_losses_t, 100)
plt.plot(train_loss_ma)
plt.ylim(train_loss_ma.min(), torch.quantile(train_loss_ma, .99))
plt.title("training loss history")
plt.show()
